# Design-choice ablation and learning curve

Two controlled experiments on the final corpus, reported as Table V and Table VII of
*Emotion Classification in Kazakh Social Media: What Limits Performance in a Low-Resource
Setting?* (ICTACS-2026), together with the breakdown of the Shame class by the presence of an
explicit lexical marker.

1. **Design-choice ablation.** Four configurations — the reported one, inference without the
   parent term, no hierarchy at all, and a cross-entropy in place of the focal loss — each
   over three seeds, with a paired bootstrap of the seed mean against the reported
   configuration. Three comparisons are made at each level, so the Bonferroni-corrected
   threshold is p < 0.0167.
2. **Learning curve.** Stratified subsamples of the authentic training pool at 25, 50, 75 and
   100 per cent, three seeds per point, with the evaluation set held fixed, which isolates
   training volume from every other factor.
3. **Shame by marker.** Per-subset precision, recall and F1 for the classifier and for a
   keyword rule, scored on identical subsets.

Input: `dataset_v20.xlsx` and `gold_v20.xlsx`. Runtime is roughly four hours on one T4 GPU;
the sections are independent once the setup cell has run.


In [ ]:
!pip install transformers torch pandas openpyxl scikit-learn -q

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd, numpy as np, random, gc, json, itertools
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.amp import autocast, GradScaler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert device.type == "cuda", "enable the GPU"
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

LABEL_MAP_L1 = {"Joy":0,"Sadness":1,"Anger":2,"Fear":3,"Surprise":4,"Disgust":5,"Neutral":6,"Shame":7}
LABEL_MAP_L2 = {"Joy":0,"Hopelessness":1,"Worthlessness":2,"Exhaustion":3,"Loneliness":4,
                "Frustration":5,"Indignation":6,"Shame":7,"Fear":8,"Surprise":9,
                "Disgust":10,"Neutral":11}
L2_TO_L1 = {"Joy":"Joy","Hopelessness":"Sadness","Worthlessness":"Sadness",
            "Exhaustion":"Sadness","Loneliness":"Sadness","Frustration":"Anger",
            "Indignation":"Anger","Shame":"Shame","Fear":"Fear","Surprise":"Surprise",
            "Disgust":"Disgust","Neutral":"Neutral"}
L2_NAMES = list(LABEL_MAP_L2.keys()); L1_NAMES = list(LABEL_MAP_L1.keys())
L2_PARENT = torch.tensor([LABEL_MAP_L1[L2_TO_L1[n]] for n in L2_NAMES])
PAR = L2_PARENT.numpy()
NUM_L1, NUM_L2 = 8, 12
SEEDS = [42, 123, 2026]

DATA = "/kaggle/input/datasets/tenajuro/datasetv20"      # <-- set this
df   = pd.read_excel(f"{DATA}/dataset_v20.xlsx")
gold = pd.read_excel(f"{DATA}/gold_v20.xlsx")
for d in (df, gold):
    d.dropna(subset=["emotion_10","text"], inplace=True)
    d["label_l1"] = d["emotion_10"].map(L2_TO_L1).map(LABEL_MAP_L1)
    d["label_l2"] = d["emotion_10"].map(LABEL_MAP_L2)
assert len(set(df["id"]) & set(gold["id"])) == 0
assert len(df) == 3073 and len(gold) == 413, f"wrong files: {len(df)} / {len(gold)}"

real   = df[~df.is_augmented].reset_index(drop=True)      # 2,054 authentic
y_gold = gold["label_l2"].values
print(f"authentic train pool {len(real)} | evaluation set {len(gold)}")

In [ ]:
class DS(torch.utils.data.Dataset):
    def __init__(s,t,l1,l2,tok): s.t,s.l1,s.l2,s.tok=t,l1,l2,tok
    def __len__(s): return len(s.t)
    def __getitem__(s,i):
        e=s.tok(str(s.t[i]),max_length=128,padding="max_length",truncation=True,return_tensors="pt")
        return {"input_ids":e["input_ids"].squeeze(),"attention_mask":e["attention_mask"].squeeze(),
                "label_l1":torch.tensor(s.l1[i]),"label_l2":torch.tensor(s.l2[i])}

class FocalLoss(nn.Module):
    def __init__(s,weight=None,gamma=2.0): super().__init__(); s.w,s.g=weight,gamma
    def forward(s,lg,y):
        ce=F.cross_entropy(lg,y,weight=s.w,reduction="none")
        return ((1-torch.exp(-ce))**s.g*ce).mean()

class Model(nn.Module):
    """use_hier_head=False -> no L1 conditioning and no L1 loss: a plain flat 12-way head."""
    def __init__(s, use_hier_head=True, n_frozen=4, dropout=0.3):
        super().__init__()
        s.use_hier_head = use_hier_head
        s.encoder = AutoModel.from_pretrained("sentence-transformers/LaBSE")
        for p in s.encoder.embeddings.parameters(): p.requires_grad = False
        for n_,p in s.encoder.named_parameters():
            if any(f"layer.{i}." in n_ for i in range(n_frozen)): p.requires_grad = False
        s.shared  = nn.Sequential(nn.Linear(768,256), nn.ReLU(), nn.Dropout(dropout))
        s.head_l1 = nn.Linear(256, NUM_L1)
        s.head_l2 = nn.Linear(256 + (NUM_L1 if use_hier_head else 0), NUM_L2)
    def forward(s, ii, am):
        h = s.encoder(input_ids=ii, attention_mask=am).last_hidden_state
        m = am.unsqueeze(-1).float()
        sh = s.shared((h*m).sum(1)/m.sum(1).clamp(min=1e-9))
        lg1 = s.head_l1(sh)
        if s.use_hier_head:
            lg2 = s.head_l2(torch.cat([sh, F.softmax(lg1,-1)], -1))
        else:
            lg2 = s.head_l2(sh)
        return lg1, lg2

def evaluate(model, loader, parent_term):
    model.eval(); par = L2_PARENT.to(device); P2,Y2,PR = [],[],[]
    with torch.no_grad():
        for b in loader:
            ii,am = b["input_ids"].to(device), b["attention_mask"].to(device)
            with autocast("cuda"): lg1,lg2 = model(ii,am)
            lp1 = F.log_softmax(lg1.float(),-1); lp2 = F.log_softmax(lg2.float(),-1)
            joint = lp2 + lp1[:,par] if parent_term else lp2
            P2.extend(joint.argmax(1).cpu().numpy()); Y2.extend(b["label_l2"].numpy())
            PR.append(torch.softmax(joint,1).cpu())
    return np.array(P2), np.array(Y2), torch.cat(PR).numpy()

def run(seed, train_df, *, use_hier_head=True, parent_term=True, use_focal=True,
        epochs=12, patience=3, bs=16, verbose=False):
    """train_df: the authentic posts to train on. validation is carved from it."""
    set_seed(seed)
    tr, va = train_test_split(train_df, test_size=0.15, stratify=train_df["label_l2"],
                              random_state=seed)
    tok = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")
    mk  = lambda d: DS(d["text"].tolist(), d["label_l1"].tolist(), d["label_l2"].tolist(), tok)
    trL = torch.utils.data.DataLoader(mk(tr), batch_size=bs, shuffle=True)
    vaL = torch.utils.data.DataLoader(mk(va), batch_size=bs)
    teL = torch.utils.data.DataLoader(mk(gold), batch_size=bs)

    model = Model(use_hier_head=use_hier_head).to(device)
    w1 = torch.tensor(compute_class_weight("balanced", classes=np.arange(NUM_L1), y=tr["label_l1"]),
                      dtype=torch.float).to(device)
    w2 = torch.tensor(compute_class_weight("balanced", classes=np.arange(NUM_L2), y=tr["label_l2"]),
                      dtype=torch.float).to(device)
    c1 = nn.CrossEntropyLoss(weight=w1)
    c2 = FocalLoss(weight=w2) if use_focal else nn.CrossEntropyLoss(weight=w2)
    enc   = [p for p in model.encoder.parameters() if p.requires_grad]
    heads = list(model.shared.parameters())+list(model.head_l1.parameters())+list(model.head_l2.parameters())
    opt = AdamW([{"params":enc,"lr":2e-5},{"params":heads,"lr":5e-4}], weight_decay=0.01)
    steps = len(trL)*epochs
    sch = get_linear_schedule_with_warmup(opt, steps//10, steps); scaler = GradScaler("cuda")

    best, best_state, wait = -1, None, 0
    for ep in range(epochs):
        model.train()
        for b in trL:
            ii,am = b["input_ids"].to(device), b["attention_mask"].to(device)
            y1,y2 = b["label_l1"].to(device), b["label_l2"].to(device)
            opt.zero_grad()
            with autocast("cuda"):
                lg1,lg2 = model(ii,am)
                loss = (0.6*c1(lg1,y1) + 0.4*c2(lg2,y2)) if use_hier_head else c2(lg2,y2)
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sch.step()
        pv,yv,_ = evaluate(model, vaL, parent_term and use_hier_head)
        f2v = f1_score(yv, pv, average="macro", zero_division=0)
        if verbose: print(f"    ep {ep+1:2d}  val F1-L2 {f2v:.4f}")
        if f2v > best: best, wait, best_state = f2v, 0, {k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    preds, labels, probs = evaluate(model, teL, parent_term and use_hier_head)
    del model; torch.cuda.empty_cache(); gc.collect()
    return preds, probs

## 1. Design-choice ablation

Four configurations, three seeds each, all trained on the same 2,054 authentic posts; only
the flag under test changes.

In [ ]:
CONFIGS = {
    "Full (hier head + parent term + focal)": dict(use_hier_head=True,  parent_term=True,  use_focal=True),
    "- parent term at inference":             dict(use_hier_head=True,  parent_term=False, use_focal=True),
    "- hierarchy entirely (flat 12-way)":     dict(use_hier_head=False, parent_term=False, use_focal=True),
    "cross-entropy instead of focal":         dict(use_hier_head=True,  parent_term=True,  use_focal=False),
}

abl_preds, abl_probs, abl_rows = {}, {}, []
for name, cfg in CONFIGS.items():
    per_seed = []
    abl_preds[name] = {}; abl_probs[name] = {}
    for s in SEEDS:
        p, pr = run(s, real, **cfg)
        abl_preds[name][s] = p; abl_probs[name][s] = pr
        f2 = f1_score(y_gold, p, average="macro", zero_division=0)
        f1_ = f1_score(PAR[y_gold], PAR[p], average="macro", zero_division=0)
        per_seed.append((f2, f1_))
        print(f"  {name:<40} seed {s}: L2 {f2:.4f} | L1 {f1_:.4f}")
    a = np.array(per_seed)
    abl_rows.append({"config": name,
                     "L2 mean": a[:,0].mean(), "L2 sd": a[:,0].std(ddof=1),
                     "L1 mean": a[:,1].mean(), "L1 sd": a[:,1].std(ddof=1)})
    slug = "".join(ch if ch.isalnum() else "_" for ch in name)[:40]
    np.save(f"/kaggle/working/ablpred_{slug}.npy", np.array([abl_preds[name][s] for s in SEEDS]))
    np.save(f"/kaggle/working/ablprob_{slug}.npy", np.array([abl_probs[name][s] for s in SEEDS]))

abl = pd.DataFrame(abl_rows)
print()
print("=== TABLE: hierarchy ablation, final corpus, 3 seeds ===")
for _, r in abl.iterrows():
    print(f'{r["config"]:<42} {r["L2 mean"]:.3f} ± {r["L2 sd"]:.3f}   {r["L1 mean"]:.3f} ± {r["L1 sd"]:.3f}')
abl.to_csv("/kaggle/working/ablation_final.csv", index=False)

In [ ]:
# paired bootstrap of every ablation row against Full, on identical resamples
RNG = np.random.default_rng(42); B = 2000
idx = [RNG.integers(0, len(y_gold), len(y_gold)) for _ in range(B)]
REF = "Full (hier head + parent term + focal)"

def seedmean_metric(name, i, level):
    vals = []
    for s in SEEDS:
        p = abl_preds[name][s]
        if level == "L2": vals.append(f1_score(y_gold[i], p[i], average="macro", zero_division=0))
        else:             vals.append(f1_score(PAR[y_gold][i], PAR[p][i], average="macro", zero_division=0))
    return float(np.mean(vals))

print("paired bootstrap vs Full (seed-mean macro-F1, 2000 resamples)")
print(f'{"config":<42}{"level":>6}{"delta":>9}{"95% CI":>20}{"p":>8}')
boot_rows = []
for name in CONFIGS:
    if name == REF: continue
    for level in ("L2", "L1"):
        d = np.array([seedmean_metric(name, i, level) - seedmean_metric(REF, i, level) for i in idx])
        lo, hi = np.percentile(d, [2.5, 97.5])
        pv = 2*min((d <= 0).mean(), (d >= 0).mean())
        flag = "significant" if lo*hi > 0 else "within CI"
        print(f'{name:<42}{level:>6}{d.mean():>+9.4f}   [{lo:+.4f}, {hi:+.4f}]  {pv:>6.3f}  {flag}')
        boot_rows.append({"config": name, "level": level, "delta": d.mean(),
                          "lo": lo, "hi": hi, "p": pv, "verdict": flag})
pd.DataFrame(boot_rows).to_csv("/kaggle/working/ablation_paired_bootstrap.csv", index=False)
print()
print("Three comparisons per level: the Bonferroni-corrected threshold is p < 0.0167.")


## 2. Learning curve

Stratified subsamples of the authentic training pool, three seeds per point. The evaluation
set is untouched, so training volume is the only factor that varies.

In [ ]:
FRACTIONS = [0.25, 0.50, 0.75, 1.00]
WATCH = ["Hopelessness", "Joy", "Frustration", "Worthlessness"]

lc_rows = []
for frac in FRACTIONS:
    for s in SEEDS:
        if frac < 1.0:
            sub, _ = train_test_split(real, train_size=frac, stratify=real["label_l2"], random_state=s)
        else:
            sub = real
        p, _ = run(s, sub.reset_index(drop=True))
        f2 = f1_score(y_gold, p, average="macro", zero_division=0)
        per = f1_score(y_gold, p, average=None, labels=range(NUM_L2), zero_division=0)
        row = {"frac": frac, "n_train": len(sub), "seed": s, "L2_macro": f2}
        for c in WATCH: row[c] = per[LABEL_MAP_L2[c]]
        lc_rows.append(row)
        print(f"  frac {frac:.2f} (n={len(sub):4d}) seed {s}: macro {f2:.4f} | " +
              " ".join(f"{c} {row[c]:.3f}" for c in WATCH))

lc = pd.DataFrame(lc_rows)
agg = lc.groupby("frac").agg(["mean", "std"]).round(4)
print()
print("=== TABLE: learning curve (3 seeds per point) ===")
print(agg[[("n_train","mean"),("L2_macro","mean"),("L2_macro","std")] +
          [(c,"mean") for c in WATCH]].to_string())
lc.to_csv("/kaggle/working/learning_curve.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"serif","font.size":9})
g = lc.groupby("frac")
fig, ax = plt.subplots(figsize=(5.2,3.2), dpi=300)
m, sd = g["L2_macro"].mean(), g["L2_macro"].std()
n = g["n_train"].mean()
ax.errorbar(n, m, yerr=sd, color="#111111", lw=1.8, marker="o", ms=5, capsize=3,
            label="12-class macro-F1")
for c, ls in zip(WATCH, [(0,(4,2)), (0,(1,1.5)), (0,(6,2,1,2)), (0,(3,1,1,1))]):
    ax.plot(n, g[c].mean(), color="#666666", lw=1.3, ls=ls, marker="s", ms=3.5, label=c)
ax.set_xlabel("authentic training posts"); ax.set_ylabel("F1 on the evaluation set")
ax.yaxis.grid(True, color="#dddddd", lw=0.5); ax.set_axisbelow(True)
for sp in ("top","right"): ax.spines[sp].set_visible(False)
ax.legend(frameon=False, fontsize=7.5, ncol=2)
fig.tight_layout(); fig.savefig("/kaggle/working/fig_learning_curve.png"); plt.show()
print("saved fig_learning_curve.png (300 dpi)")

## 3. Shame by presence of an explicit marker

Reuses the predictions of the reported configuration. Of the 33 Shame items in the evaluation
set, 31 carry an explicit marker and 2 do not, so the implicit case cannot be estimated; it is
reported as such rather than as a score. The keyword rule is scored on identical subsets.

In [ ]:
ROOTS_KZ = ["ұя", "уя"]                       # ұят, ұялу, ұялдым ...
ROOTS_RU = ["стыд", "позор", "неловк", "стесня", "кринж"]
def has_marker(t):
    s = str(t).lower()
    return any(r in s for r in ROOTS_KZ) or any(r in s for r in ROOTS_RU)

gold = gold.reset_index(drop=True)
mark = gold["text"].map(has_marker).values
SH = LABEL_MAP_L2["Shame"]
is_shame = (y_gold == SH)

print(f"evaluation set: {len(gold)} posts, {is_shame.sum()} labelled Shame")
print(f"  Shame with an explicit marker : {(is_shame & mark).sum()}")
print(f"  Shame without                 : {(is_shame & ~mark).sum()}   <- not estimable")
print(f"  non-Shame carrying a marker   : {(~is_shame & mark).sum()} "
      f"({(~is_shame & mark).sum()/(~is_shame).sum()*100:.1f} % of non-Shame)")
print()

def prf(y_true_bin, y_pred_bin):
    tp = int((y_true_bin & y_pred_bin).sum()); fp = int((~y_true_bin & y_pred_bin).sum())
    fn = int((y_true_bin & ~y_pred_bin).sum())
    p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f, tp, fp, fn

ens = np.mean([abl_probs[REF][s] for s in SEEDS], axis=0).argmax(1)   # probability-averaged ensemble
systems = {"hierarchical classifier (ensemble)": (ens == SH),
           "keyword rule (marker -> Shame)":      mark}

print(f'{"system":<36}{"subset":<22}{"P":>7}{"R":>7}{"F1":>7}{"n":>5}')
rows = []
for sysname, pred in systems.items():
    for subset, m in [("all Shame items", np.ones(len(gold), bool)),
                      ("explicit marker only", mark | ~is_shame),
                      ("implicit only", (~mark) | ~is_shame)]:
        yt = is_shame & m; yp = pred & m
        p, r, f, tp, fp, fn = prf(yt, yp)
        note = "  (n<5: not estimable)" if yt.sum() < 5 else ""
        print(f'{sysname:<36}{subset:<22}{p:>7.3f}{r:>7.3f}{f:>7.3f}{int(yt.sum()):>5}{note}')
        rows.append({"system": sysname, "subset": subset, "P": p, "R": r, "F1": f,
                     "support": int(yt.sum()), "tp": tp, "fp": fp, "fn": fn})
pd.DataFrame(rows).to_csv("/kaggle/working/shame_explicit_vs_implicit.csv", index=False)
print()
